# Beauty Products Data Pipeline

Extract makeup and perfume data from two public APIs, clean it with Pandas, load it into SQLite, and export it to Excel.

- **Makeup API** — makeup products from real brands, with prices, tags, and shades
- **Open Beauty Facts** — open database of beauty products (perfumes and makeup), licensed under ODbL

## 1. Imports

In [1]:
import os
import re
import html
import time
import sqlite3
import requests
import pandas as pd

In [2]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)

## 2. Extract — Makeup API

In [3]:
makeup_url = "https://makeup-api.herokuapp.com/api/v1/products.json"
response = requests.get(makeup_url, timeout=120)
makeup_data = response.json()
len(makeup_data)

931

In [4]:
id_list = []
brand_list = []
name_list = []
type_list = []
category_list = []
price_list = []
currency_list = []
rating_list = []
link_list = []
image_list = []
for product in makeup_data:
    id_list.append(product["id"])
    brand_list.append(product["brand"])
    name_list.append(product["name"])
    type_list.append(product["product_type"])
    category_list.append(product["category"])
    price_list.append(product["price"])
    currency_list.append(product["currency"])
    rating_list.append(product["rating"])
    try:
        link_list.append(product["product_link"])
    except:
        link_list.append("")

    try:
        image_list.append(product["image_link"])
    except:
        image_list.append("")

In [5]:
color_product_id_list = []
color_name_list = []
hex_list = []
for product in makeup_data:
    for color in product["product_colors"]:
        color_product_id_list.append(product["id"])
        color_name_list.append(color.get("colour_name", ""))
        hex_list.append(color.get("hex_value", ""))

In [6]:
tag_product_id_list = []
tag_list = []
for product in makeup_data:
    for tag in product["tag_list"]:
        tag_product_id_list.append(product["id"])
        tag_list.append(tag)

In [7]:
df_makeup = pd.DataFrame({
    "product_id": id_list,
    "brand": brand_list,
    "name": name_list,
    "product_type": type_list,
    "category": category_list,
    "price": price_list,
    "currency": currency_list,
    "rating": rating_list,
    "product_link": link_list,
    "image_link": image_list,
})

df_colors = pd.DataFrame({
    "product_id": color_product_id_list,
    "color_name": color_name_list,
    "hex_value": hex_list,
})

df_tags = pd.DataFrame({
    "product_id": tag_product_id_list,
    "tag": tag_list,
})

In [8]:
len(df_makeup), len(df_colors), len(df_tags)

(931, 5329, 419)

## 3. Extract — Open Beauty Facts

The search API allows about 10 requests per minute, so the loop waits 7 seconds between pages.

In [9]:
obf_url = "https://world.openbeautyfacts.org/api/v2/search"
headers = {"User-Agent": "BeautyProductsPipeline/1.0 (https://github.com/nepal-aly)"}
fields = "code,product_name,brands,countries_tags,quantity,image_url"
categories = ["perfumes", "makeup"]

In [10]:
obf_products = []
for category in categories:
    page = 1
    while True:
        params = {
            "categories_tags_en": category,
            "page_size": 100,
            "page": page,
            "fields": fields,
        }
        response = requests.get(obf_url, params=params, headers=headers, timeout=120)
        products = response.json()["products"]
        for product in products:
            product["search_category"] = category
            obf_products.append(product)

        print(f"{category} - page {page}: {len(products)} products")
        if len(products) < 100:
            break
        page += 1
        time.sleep(7)
    time.sleep(7)

perfumes - page 1: 100 products


perfumes - page 2: 100 products


perfumes - page 3: 100 products


perfumes - page 4: 100 products


perfumes - page 5: 48 products


...

makeup - page 6: 100 products


makeup - page 7: 100 products


makeup - page 8: 100 products


makeup - page 9: 100 products


makeup - page 10: 52 products


In [11]:
code_list = []
obf_name_list = []
obf_brand_list = []
obf_category_list = []
country_list = []
quantity_list = []
obf_image_list = []
for product in obf_products:
    code_list.append(product["code"])
    obf_category_list.append(product["search_category"])
    try:
        obf_name_list.append(product["product_name"])
    except:
        obf_name_list.append("")

    try:
        obf_brand_list.append(product["brands"])
    except:
        obf_brand_list.append("")

    try:
        country_list.append(", ".join(product["countries_tags"]))
    except:
        country_list.append("")

    try:
        quantity_list.append(product["quantity"])
    except:
        quantity_list.append("")

    try:
        obf_image_list.append(product["image_url"])
    except:
        obf_image_list.append("")

In [12]:
df_obf = pd.DataFrame({
    "code": code_list,
    "name": obf_name_list,
    "brand": obf_brand_list,
    "category": obf_category_list,
    "countries": country_list,
    "quantity": quantity_list,
    "image_url": obf_image_list,
})
len(df_obf)

1400

In [13]:
df_makeup.to_csv("data/raw/makeup_products_raw.csv", index=False)
df_colors.to_csv("data/raw/makeup_colors_raw.csv", index=False)
df_tags.to_csv("data/raw/makeup_tags_raw.csv", index=False)
df_obf.to_csv("data/raw/open_beauty_facts_raw.csv", index=False)

## 4. Transform — Data quality before cleaning

In [14]:
quality_before = pd.DataFrame({
    "missing_brand": [df_makeup["brand"].isna().sum()],
    "missing_price": [df_makeup["price"].isna().sum()],
    "zero_price": [(df_makeup["price"] == "0.0").sum()],
    "missing_currency": [df_makeup["currency"].isna().sum()],
    "missing_rating": [df_makeup["rating"].isna().sum()],
    "obf_missing_name": [(df_obf["name"].fillna("").str.strip() == "").sum()],
    "obf_missing_brand": [(df_obf["brand"].fillna("").str.strip() == "").sum()],
    "obf_duplicate_codes": [df_obf["code"].duplicated().sum()],
})
display(quality_before)

,missing_brand,missing_price,zero_price,missing_currency,missing_rating,obf_missing_name,obf_missing_brand,obf_duplicate_codes
0,12,14,40,563,591,92,194,0


## 5. Transform — Makeup products

In [15]:
brand_fix = {
    "Nyx": "NYX",
    "E.L.F.": "e.l.f.",
    "L'Oreal": "L'Oréal",
    "Iman": "IMAN",
    "W3Llpeople": "W3LL People",
    "Covergirl": "CoverGirl",
}

df_makeup_clean = df_makeup.copy()
df_makeup_clean["brand"] = df_makeup_clean["brand"].fillna("unknown").str.strip().str.title()
df_makeup_clean["brand"] = df_makeup_clean["brand"].replace(brand_fix)
df_makeup_clean["name"] = df_makeup_clean["name"].apply(html.unescape).str.strip()
df_makeup_clean["product_type"] = df_makeup_clean["product_type"].str.replace("_", " ").str.title()
df_makeup_clean["category"] = df_makeup_clean["category"].fillna("other").str.replace("_", " ").str.title()
df_makeup_clean["price"] = pd.to_numeric(df_makeup_clean["price"], errors="coerce")
df_makeup_clean.loc[df_makeup_clean["price"] == 0, "price"] = None
df_makeup_clean["currency"] = df_makeup_clean["currency"].fillna("Unknown")
df_makeup_clean["rating"] = pd.to_numeric(df_makeup_clean["rating"], errors="coerce")
df_makeup_clean = df_makeup_clean.drop_duplicates(subset=["brand", "name", "product_type"])
df_makeup_clean.head()

,product_id,brand,name,product_type,category,price,currency,rating,product_link,image_link
0,1048,Colourpop,Lippie Pencil,Lip Liner,Pencil,5.0,CAD,NaN,https://colourpop.com/collections/lippie-pencil,https://cdn.shopify.com/s/files/1/1338/0845/co...
1,1047,Colourpop,Blotted Lip,Lipstick,Lipstick,5.5,CAD,NaN,https://colourpop.com/collections/lippie-stix?...,https://cdn.shopify.com/s/files/1/1338/0845/pr...
2,1046,Colourpop,Lippie Stix,Lipstick,Lipstick,5.5,CAD,NaN,https://colourpop.com/collections/lippie-stix,https://cdn.shopify.com/s/files/1/1338/0845/co...
3,1045,Colourpop,No Filter Foundation,Foundation,Liquid,12.0,CAD,NaN,https://colourpop.com/products/no-filter-matte...,https://cdn.shopify.com/s/files/1/1338/0845/pr...
4,1044,Boosh,Lipstick,Lipstick,Lipstick,26.0,CAD,NaN,https://www.boosh.ca/collections/all,https://cdn.shopify.com/s/files/1/1016/3243/pr...


## 6. Transform — Shades and tags

In [16]:
df_colors_clean = df_colors.copy()
df_colors_clean["color_name"] = df_colors_clean["color_name"].fillna("").str.strip()
df_colors_clean["hex_value"] = df_colors_clean["hex_value"].fillna("").str.strip().str.lower()
valid_hex = df_colors_clean["hex_value"].str.match(r"^#[0-9a-f]{6}$")
df_colors_clean = df_colors_clean[valid_hex]
df_colors_clean = df_colors_clean[df_colors_clean["product_id"].isin(df_makeup_clean["product_id"])]
df_colors_clean = df_colors_clean.drop_duplicates()
len(df_colors), len(df_colors_clean)

(5329, 5254)

In [17]:
df_tags_clean = df_tags.copy()
tag_fix = {
    "Ewg Verified": "EWG Verified",
    "Certclean": "CertClean",
    "Usda Organic": "USDA Organic",
    "Non-Gmo": "Non-GMO",
    "Ecocert": "EcoCert",
}

df_tags_clean["tag"] = df_tags_clean["tag"].str.strip().str.title()
df_tags_clean["tag"] = df_tags_clean["tag"].replace(tag_fix)
df_tags_clean = df_tags_clean[df_tags_clean["tag"] != "Purpicks"]
df_tags_clean = df_tags_clean[df_tags_clean["product_id"].isin(df_makeup_clean["product_id"])]
df_tags_clean = df_tags_clean.drop_duplicates()
df_tags_clean["tag"].value_counts().head(10)

tag
Natural           82
Gluten Free       81
Vegan             66
Canadian          51
EWG Verified      15
Organic           14
CertClean         10
Hypoallergenic     8
No Talc            8
USDA Organic       7
Name: count, dtype: int64

## 7. Transform — Open Beauty Facts

In [18]:
def quantity_to_amount(quantity, category):
    text = str(quantity).lower().replace(",", ".")
    match = re.search(r"(\d+(?:\.\d+)?)\s*(fl\.?\s*oz|ml|cl|l|kg|mg|g|oz)\b", text)
    if not match:
        return None, ""
    value = float(match.group(1))
    unit = match.group(2).replace(" ", "").replace(".", "")
    if unit == "ml":
        return value, "ml"
    if unit == "cl":
        return value * 10, "ml"
    if unit == "l":
        return value * 1000, "ml"
    if unit == "floz":
        return round(value * 29.57, 1), "ml"
    if unit == "g":
        return value, "g"
    if unit == "kg":
        return value * 1000, "g"
    if unit == "mg":
        return value / 1000, "g"
    if unit == "oz" and category == "Perfumes":
        return round(value * 29.57, 1), "ml"
    if unit == "oz":
        return round(value * 28.35, 1), "g"
    return None, ""

In [19]:
df_obf_clean = df_obf.copy()
df_obf_clean["name"] = df_obf_clean["name"].fillna("").str.strip()
df_obf_clean = df_obf_clean[df_obf_clean["name"] != ""]
df_obf_clean["brand"] = df_obf_clean["brand"].fillna("").str.split(",").str[0].str.strip().str.title()
df_obf_clean.loc[df_obf_clean["brand"] == "", "brand"] = "Unknown"
df_obf_clean["category"] = df_obf_clean["category"].str.title()
df_obf_clean["countries"] = df_obf_clean["countries"].str.replace(r"[a-z]{2}:", "", regex=True).str.replace("-", " ").str.title()

amount_list = []
unit_list = []
for quantity, category in zip(df_obf_clean["quantity"], df_obf_clean["category"]):
    amount, unit = quantity_to_amount(quantity, category)
    amount_list.append(amount)
    unit_list.append(unit)
df_obf_clean["amount"] = amount_list
df_obf_clean["unit"] = unit_list

df_obf_clean = df_obf_clean.drop_duplicates(subset=["code"])
df_obf_clean.head()

,code,name,brand,category,countries,quantity,image_url,amount,unit
0,5901887045243,Jeju młoda skóra,Ziaja,Perfumes,Poland,200 ml,https://images.openbeautyfacts.org/images/prod...,200.0,ml
1,0079400511041,Axe: Black Vanilla Orange + Sandlewood,Axe,Perfumes,United States,1 X 1oz,,29.6,ml
2,8402001027994,Mgiełka pistacjowa do ciała i włosów Deliplus ...,Deliplus,Perfumes,"Poland, Spain",250 ml,,250.0,ml
3,6291110109033,French Coffee,Al-Rehab,Perfumes,Poland,6 ml,https://images.openbeautyfacts.org/images/prod...,6.0,ml
4,3596710556878,Eau de cologne natural,Cosmia,Perfumes,France,0.25 l,https://images.openbeautyfacts.org/images/prod...,250.0,ml


## 8. Check the clean data

In [20]:
len(df_makeup_clean), len(df_colors_clean), len(df_tags_clean), len(df_obf_clean)

(916, 5254, 378, 1308)

In [21]:
quality_after = pd.DataFrame({
    "table": ["makeup_products", "makeup_colors", "makeup_tags", "beauty_products"],
    "rows_before": [len(df_makeup), len(df_colors), len(df_tags), len(df_obf)],
    "rows_after": [len(df_makeup_clean), len(df_colors_clean), len(df_tags_clean), len(df_obf_clean)],
})
quality_after["rows_removed"] = quality_after["rows_before"] - quality_after["rows_after"]
display(quality_after)

,table,rows_before,rows_after,rows_removed
0,makeup_products,931,916,15
1,makeup_colors,5329,5254,75
2,makeup_tags,419,378,41
3,beauty_products,1400,1308,92


## 9. Load — SQLite database

In [22]:
connection = sqlite3.connect("data/clean/beauty_products.db")
df_makeup_clean.to_sql("makeup_products", connection, if_exists="replace", index=False)
df_colors_clean.to_sql("makeup_colors", connection, if_exists="replace", index=False)
df_tags_clean.to_sql("makeup_tags", connection, if_exists="replace", index=False)
df_obf_clean.to_sql("beauty_products", connection, if_exists="replace", index=False)

1308

## 10. Analysis with SQL

In [23]:
query = '''
SELECT product_type, COUNT(*) AS products
FROM makeup_products
GROUP BY product_type
ORDER BY products DESC
'''
display(pd.read_sql(query, connection))

,product_type,products
0,Foundation,165
1,Lipstick,154
2,Eyeliner,145
3,Mascara,92
4,Eyeshadow,83
5,Blush,74
6,Bronzer,67
7,Nail Polish,59
8,Eyebrow,49
9,Lip Liner,28


In [24]:
query = '''
SELECT brand, COUNT(*) AS products
FROM makeup_products
GROUP BY brand
ORDER BY products DESC
LIMIT 10
'''
display(pd.read_sql(query, connection))

,brand,products
0,NYX,163
1,Clinique,93
2,Dior,74
3,Maybelline,52
4,CoverGirl,49
5,Smashbox,46
6,L'Oréal,46
7,Benefit,41
8,Physicians Formula,40
9,Revlon,29


In [25]:
query = '''
SELECT product_type, ROUND(AVG(price), 2) AS avg_price_usd, COUNT(*) AS products
FROM makeup_products
WHERE currency = 'USD' AND price IS NOT NULL
GROUP BY product_type
ORDER BY avg_price_usd DESC
'''
display(pd.read_sql(query, connection))

,product_type,avg_price_usd,products
0,Eyeshadow,20.92,13
1,Foundation,19.92,69
2,Bronzer,16.08,12
3,Blush,15.67,17
4,Eyebrow,11.30,24
5,Lipstick,11.13,55
6,Eyeliner,10.48,41
7,Lip Liner,8.44,8
8,Mascara,8.32,11


In [26]:
query = '''
SELECT p.brand, p.name, p.product_type, COUNT(c.hex_value) AS shades
FROM makeup_products p
JOIN makeup_colors c ON p.product_id = c.product_id
GROUP BY p.product_id
ORDER BY shades DESC
LIMIT 10
'''
display(pd.read_sql(query, connection))

,brand,name,product_type,shades
0,Smashbox,Be Legendary Lipstick,Lipstick,120
1,NYX,Extra Creamy Round Lipstick,Lipstick,86
2,Essie,Essie Nail Colour,Nail Polish,63
3,NYX,Slim Lip Pencil,Lip Liner,50
4,Colourpop,Lippie Stix,Lipstick,46
5,NYX,Matte Lipstick,Lipstick,45
6,Colourpop,No Filter Foundation,Foundation,42
7,Fenty,PRO FILT'R - Soft Matte Longwear Foundation,Foundation,40
8,Clinique,All About Shadow™ Quad,Eyeshadow,37
9,NYX,Suede Matte Lip Liner,Lip Liner,36


In [27]:
query = '''
SELECT p.product_type, COUNT(*) AS vegan_products
FROM makeup_products p
JOIN makeup_tags t ON p.product_id = t.product_id
WHERE t.tag = 'Vegan'
GROUP BY p.product_type
ORDER BY vegan_products DESC
'''
display(pd.read_sql(query, connection))

,product_type,vegan_products
0,Eyeshadow,12
1,Eyeliner,9
2,Nail Polish,8
3,Lipstick,8
4,Foundation,8
5,Bronzer,7
6,Mascara,6
7,Blush,5
8,Lip Liner,3


In [28]:
query = '''
SELECT brand, COUNT(*) AS perfumes, ROUND(AVG(CASE WHEN unit = 'ml' THEN amount END), 1) AS avg_size_ml
FROM beauty_products
WHERE category = 'Perfumes' AND brand != 'Unknown'
GROUP BY brand
ORDER BY perfumes DESC
LIMIT 10
'''
display(pd.read_sql(query, connection))

,brand,perfumes,avg_size_ml
0,Fragonard,14,23.0
1,Guerlain,8,95.8
2,Calvin Klein,8,126.2
3,Bee Beauty,8,233.8
4,Yves Saint Laurent,7,85.0
5,Yves Rocher,7,82.5
6,Jeanne Arthes,7,100.0
7,Cosmia,7,250.0
8,Chanel,7,116.4
9,Armaf,7,115.0


## 11. Export to Excel

In [29]:
with pd.ExcelWriter("data/clean/beauty_products.xlsx") as writer:
    df_makeup_clean.to_excel(writer, sheet_name="makeup_products", index=False)
    df_colors_clean.to_excel(writer, sheet_name="makeup_colors", index=False)
    df_tags_clean.to_excel(writer, sheet_name="makeup_tags", index=False)
    df_obf_clean.to_excel(writer, sheet_name="beauty_products", index=False)
    quality_after.to_excel(writer, sheet_name="data_quality", index=False)

In [30]:
connection.close()